In [48]:
import wrds
import pandas as pd

In [49]:
def calculate_ratio(numerator, denominator, ratio_name):
    """Helper function to calculate ratio and handle division by zero."""
    if denominator == 0:
        print(f"{ratio_name}: N/A (Denominator is zero)")
        return None
    ratio = numerator / denominator
    print(f"{ratio_name}: {ratio:.4f} ({ratio*100:.2f}%)")
    return ratio

In [50]:
# Step 1: Interactive User Input
ticker_input = input("Please enter the company ticker symbol (e.g., AAPL): ").strip().upper()
year_input = input("Please enter the year (e.g., 2023): ").strip()

start_date = f"{year_input}-01-01"
end_date = f"{year_input}-12-31"

print(f"\n--- Analyzing {ticker_input} for the year {year_input} ---")

Please enter the company ticker symbol (e.g., AAPL):  AAPL
Please enter the year (e.g., 2023):  2022



--- Analyzing AAPL for the year 2022 ---


In [51]:
# Step 2: Connect to WRDS
try:
    db = wrds.Connection()
except Exception as e:
    print(f"Connection failed: {e}")
    exit()

Enter your WRDS username [15751]: zmhzmh
Enter your password: ········


WRDS recommends setting up a .pgpass file.


Create .pgpass file now [y/n]?:  y


pgpass file created at C:\Users\15751\AppData\Roaming\postgresql\pgpass.conf
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [52]:
# Step 3: Define SQL Query with additional fields
# Added: 'teq' (Total Equity), 'act' (Current Assets), 'lct' (Current Liabilities), 'invt' (Inventory)
sql_query = f"""
SELECT 
    gvkey, 
    datadate, 
    tic,
    ni AS net_profit,          -- Net Income
    at AS total_asset,         -- Total Assets
    teq AS total_equity,       -- Total Equity
    act AS current_assets,     -- Current Assets
    lct AS current_liabilities,-- Current Liabilities
    invt AS inventory          -- Inventory
FROM 
    comp.funda
WHERE 
    tic = '{ticker_input}' 
    AND datafmt = 'STD' 
    AND indfmt = 'INDL' 
    AND popsrc = 'D' 
    AND datadate >= '{start_date}'
    AND datadate <= '{end_date}'
ORDER BY 
    datadate DESC
LIMIT 1
"""

In [53]:
# Step 4: Execute Query and Calculate Ratios
print("Executing query...")
df = db.raw_sql(sql_query, coerce_float=True)

if not df.empty:
    # Extract data from the first row
    data = df.iloc[0]
        
    print(f"\n--- Financial Data for {data['tic']} ({data['datadate']}) ---")
    print(f"Net Profit: {data['net_profit']:,.2f}")
    print(f"Total Assets: {data['total_asset']:,.2f}")
    print(f"Total Equity: {data['total_equity']:,.2f}")
    print(f"Current Assets: {data['current_assets']:,.2f}")
    print(f"Current Liabilities: {data['current_liabilities']:,.2f}")
    print(f"Inventory: {data['inventory']:,.2f}")

    print("\n--- Calculated Ratios ---")
        
    # 1. ROA = Net Income / Total Assets
    calculate_ratio(data['net_profit'], data['total_asset'], "ROA")

    # 2. ROE = Net Income / Total Equity
    calculate_ratio(data['net_profit'], data['total_equity'], "ROE")

    # 3. Current Ratio = Current Assets / Current Liabilities
    calculate_ratio(data['current_assets'], data['current_liabilities'], "Current Ratio")

    # 4. Quick Ratio = (Current Assets - Inventory) / Current Liabilities
    quick_assets = data['current_assets'] - data['inventory']
    calculate_ratio(quick_assets, data['current_liabilities'], "Quick Ratio")

else:
    print(f"\nNo data found for ticker '{ticker_input}' in year '{year_input}'.")

Executing query...

--- Financial Data for AAPL (2022-09-30) ---
Net Profit: 99,803.00
Total Assets: 352,755.00
Total Equity: 50,672.00
Current Assets: 135,405.00
Current Liabilities: 153,982.00
Inventory: 4,946.00

--- Calculated Ratios ---
ROA: 0.2829 (28.29%)
ROE: 1.9696 (196.96%)
Current Ratio: 0.8794 (87.94%)
Quick Ratio: 0.8472 (84.72%)


In [54]:
# Step 5: Financial Indicator Evaluation Function

# 1. ROA Return on Assets
def evaluate_roa(roa):
    print("\n===== ROA (Return on Assets) comment =====")
    if roa > 0.08:
        print("excellent：>8%, High asset utilization efficiency and sound overall profit quality.")
    elif 0.03 <= roa <= 0.08:
        print("average：3%~8%, Stable asset returns, staying at the industrial average level.")
    else:
        print("poor：<3%, Bulky and underutilized assets, weak profitability or even net losses.")

# 2. ROE Return on Equity
def evaluate_roe(roe):
    print("\n===== ROE (Return on Equity) comment =====")
    if roe >= 0.15:
        print("excellent：≥15%, High shareholder returns and strong competitive moat in profitability.")
    elif 0.08 <= roe < 0.15:
        print("average：8%~15%, Moderate returns, in line with the average market payoff level.")
    else:
        print("poor：<8%, Weak profit-generating capacity of internal capital and low investment value.")

# 3. Current Ratio
def evaluate_current_ratio(cr):
    print("\n===== Current Ratio comment =====")
    if cr >= 2.0:
        print("excellent：≥2.0, Sufficient current assets with extremely low short-term default risks.")
    elif 1.2 <= cr < 2.0:
        print("average：1.2~2.0, Healthy debt structure and manageable financial risks.")
    else:
        print("poor：<1.2, Heavy short-term debt pressure and high risks of capital chain tension.")

# 4. Quick Ratio
def evaluate_quick_ratio(qr):
    print("\n===== Quick Ratio comment =====")
    if qr >= 1.0:
        print("excellent：≥1.0, Able to repay short-term debts without inventory liquidation, ensuring stable cash flow.")
    elif 0.6 <= qr < 1.0:
        print("average：0.6~1.0, Basically controllable short-term solvency.")
    else:
        print("poor：<0.6, Severe short-term debt pressure and high risk of capital chain rupture.")

# Enter the company data you need to analyze here
roa = 0.2829
roe = 1.9696
current_ratio = 0.8947
quick_ratio = 0.8472

# Run all evaluations with one click
evaluate_roa(roa)
evaluate_roe(roe)
evaluate_current_ratio(current_ratio)
evaluate_quick_ratio(quick_ratio)


===== ROA (Return on Assets) comment =====
excellent：>8%, High asset utilization efficiency and sound overall profit quality.

===== ROE (Return on Equity) comment =====
excellent：≥15%, High shareholder returns and strong competitive moat in profitability.

===== Current Ratio comment =====
poor：<1.2, Heavy short-term debt pressure and high risks of capital chain tension.

===== Quick Ratio comment =====
average：0.6~1.0, Basically controllable short-term solvency.
